# Week 12, Day 2: Gradient descent: step size, convergence and divergence

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A reusable gradient descent

Write `gradient_descent(grad, x0, lr, steps)` that returns an array of all iterates (row k is the parameters after k steps). Use it on the long-valley quadratic below with three learning rates and plot the loss against iteration on a log scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gradient_descent(grad, x0, lr, steps):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(steps):
        ...
        history.append(x.copy())
    return np.array(history)

H = np.diag([10.0, 1.0])                    # curvatures 10 and 1
loss = lambda w: 0.5 * w @ H @ w
grad = lambda w: H @ w
plt.figure(figsize=(5, 3.2))
for lr in [0.02, 0.1, 0.18, 0.21]:
    h = gradient_descent(grad, [1.0, 1.0], lr, 60)
    plt.semilogy([loss(w) for w in h], label=f"η = {lr}")
plt.ylim(1e-12, 1e4); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(frameon=False); plt.show()

In [ ]:
import numpy as np
def test_history():
    "gradient_descent returns an array of steps + 1 iterates starting at x0"
    h = gradient_descent(lambda x: 2 * x, np.array([1.0, -2.0]), 0.1, 5)
    assert h.shape == (6, 2) and np.allclose(h[0], [1, -2])

def test_quadratic():
    "on f = x^2 with lr 0.1 each step multiplies x by 0.8"
    h = gradient_descent(lambda x: 2 * x, np.array([1.0]), 0.1, 3)
    assert np.allclose(h[:, 0], [1, 0.8, 0.64, 0.512])

test_history()
test_quadratic()
print("All checks passed")

### Exercise: Measure the convergence rate

For Hessians diag(κ, 1) with κ = 2, 10 and 100, run gradient descent with the best learning rate $2/(\kappa + 1)$ from (1, 1). Measure the contraction per step as the ratio of the distance to the minimum at steps 31 and 30 (`measured`), and count the steps until that distance falls below 1e-6 (`steps_needed`).

In [ ]:
kappas = [2.0, 10.0, 100.0]
measured, steps_needed = [], []
for kap in kappas:
    Hk = np.diag([kap, 1.0])
    lr = ...
    h = gradient_descent(lambda w: Hk @ w, [1.0, 1.0], lr, 3000)
    dist = np.linalg.norm(h, axis=1)
    measured.append(...)
    steps_needed.append(int(np.argmax(dist < 1e-6)))
print("contraction per step:", np.round(measured, 4), " predicted:", np.round([(k - 1) / (k + 1) for k in kappas], 4))
print("steps for a millionfold reduction:", steps_needed)

In [ ]:
import numpy as np
def test_rate():
    "the measured contraction per step at the best learning rate matches (kappa - 1) / (kappa + 1)"
    for kap, r in zip(kappas, measured):
        assert abs(r - (kap - 1) / (kap + 1)) < 0.01, (kap, r)

def test_steps():
    "steps to reduce the error a millionfold grow roughly in proportion to kappa"
    assert 7 < steps_needed[2] / steps_needed[1] < 13

test_rate()
test_steps()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(12)
theta = np.repeat(np.linspace(0, 2 * np.pi, 16, endpoint=False), 10)
def von_mises(theta, mu, kappa, b, A):
    return b + A * np.exp(kappa * (np.cos(theta - mu) - 1))
counts = rng.poisson(von_mises(theta, 1.0, 2.0, 5.0, 30.0))

def nll(p):
    mu, log_k, log_b, log_A = p
    lam = von_mises(theta, mu, np.exp(log_k), np.exp(log_b), np.exp(log_A))
    return np.mean(lam - counts * np.log(lam))

def numerical_grad(f, p, h=1e-6):
    g = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p); e[i] = h
        g[i] = (f(p + e) - f(p - e)) / (2 * h)
    return g
print("NLL at a rough guess:", round(nll(np.array([0.0, 0.0, 1.0, 2.0])), 3))

### Exercise: Fit four parameters by gradient descent

Starting from the rough guess (μ = 0, κ = 1, baseline e, amplitude e²), run 4000 steps of gradient descent with learning rate 0.02 on `nll`, using `numerical_grad`. Record the NLL at every iterate in `nll_hist`, and convert the final parameters back to (μ, κ, b, A) in `fitted`.

In [ ]:
p0 = np.array([0.0, 0.0, 1.0, 2.0])          # mu, log kappa, log baseline, log amplitude
hist = ...
nll_hist = [nll(p) for p in hist]
mu, log_k, log_b, log_A = hist[-1]
fitted = ...

grid = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(nll_hist); ax[0].set_xlabel("step"); ax[0].set_ylabel("Poisson NLL")
ax[1].plot(np.degrees(theta), counts, ".", alpha=0.3); ax[1].plot(np.degrees(grid), von_mises(grid, *fitted), lw=2); ax[1].set_xlabel("direction (deg)")
plt.tight_layout(); plt.show()
print("fitted (mu, kappa, b, A):", np.round(fitted, 3))

In [ ]:
import numpy as np
def test_decrease():
    "the NLL falls over training"
    assert nll_hist[-1] < nll_hist[0] - 1
    assert len(nll_hist) == 4001

def test_params():
    "the fitted parameters are close to the truth (mu 1.0, kappa 2, baseline 5, amplitude 30)"
    mu, k, b, A = fitted
    assert abs(np.angle(np.exp(1j * (mu - 1.0)))) < 0.1 and abs(k - 2) < 0.5 and abs(b - 5) < 1.5 and abs(A - 30) < 4

test_decrease()
test_params()
print("All checks passed")